# 06 · Validez predictiva

Fase CRISP-DM **Evaluation**. La cuarta pregunta del proyecto: ¿la predicción
**aguanta fuera de la muestra** con la que se ajustó?

| Pregunta | Módulo |
|---|---|
| ¿El dato tiene nulos, rangos y unicidad? | `etl-project/src/quality/dimensions.py` |
| ¿Las métricas del ETL son defendibles? | `etl-project/src/quality/auditoria.py` |
| ¿Las fuentes ONSV/SUTRAN/OSITRAN son de fiar? | `05_fiabilidad_fuentes.ipynb` |
| **¿La predicción aguanta fuera de muestra?** | **este notebook** |

Un modelo puede tener un R² alto dentro de muestra y ser inútil: eso es
sobreajuste. Y puede tener un R² negativo y seguir siendo útil para lo único
que se le pide — **ordenar** tramos por riesgo. Este notebook separa las dos
cuestiones y no las mezcla en un veredicto único.

Todo lo que sigue lo calcula `scripts/validez_predictiva.py`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'scripts'))

DATA = ROOT / 'data' / 'processed'

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 170)
pd.set_option('display.max_colwidth', 90)
import validez_predictiva as vp

print(vp.__doc__.strip().splitlines()[0])

# `analizar()` recalcula los dos protocolos; no lee el JSON que otro dejo
# escrito. Es lo que hace que este notebook pueda contradecir al script.
res = vp.analizar(verbose=False)
print('\nClaves del informe:', list(res))

## 6.1 · Protocolo espacial: ¿usar las covariables mejora algo?

Cross-validation agrupado por corredor. Si el despliegue predice tramos que
nunca vio en entrenamiento, la validación tiene que ser agrupada: con k-fold
plano, tramos vecinos del mismo corredor caen en entrenamiento y en prueba a
la vez, y el modelo se evalúa con información que en producción no tendría.

In [ ]:
pe = res['protocolo_espacial']
print('tipo       :', pe['tipo'])
print('agrupacion :', pe['agrupacion'])
print('corredores :', pe['n_corredores'])
print('folds      :', pe['folds'])

print(pe['agregado'])

### La mejora es real pero pequeña, y el R² sigue siendo negativo

La devianza de Poisson (menor es mejor) es la métrica principal aquí, porque
el objetivo es un **conteo**: la media y el MAE premian predecir cero, que es
lo que hace el 86 % de las celdas. Un R² sobre un conteo que casi siempre es 0
mide la varianza de unos pocos tramos con evento, no capacidad de predecir.

## 6.2 · Protocolo temporal: holdout de 2025

El segundo Split es el que responde a la pregunta del título: se entrena con
2021-2024 y se prueba con 2025, que el modelo **nunca** vio. Es la simulación de
lo que pasa al desplegar en el future.

In [ ]:
pt = res['protocolo_temporal']
print(f"Entrenamiento : {pt['entrenamiento']}")
print(f"Holdout       : {pt['holdout']}")
print(f"Celdas test   : {pt['n_test']}   eventos en test: {pt['eventos_test']}")

tabla = pd.DataFrame(pt['modelos']).T
print(tabla.to_string())

### Por qué aquí no se declara un ganador

Cada métrica favorece a un modelo distinto:

- por **MAE** gana la persistencia (predecir el valor del año anterior),
- por **devianza de Poisson** gana el NegBin frente a la media.

No es que un modelo sea mejor: es que **las métricas no son equivalentes en
una matriz con 86 % de ceros**, y elegir con estos datos sería arbitrario. El
veredicto se declara `AMBIGUO` y se explica por qué, en vez de publishes un
número que favours a un modelo por accidente.

In [ ]:
for k, v in res['notas'].items():
    print(f'[{k}] {v}\n')

## 6.3 · Intervalos: la cobertura no es la nominal

Un intervalo al 95 % que cubre el 91 % de los casos **subestima** la
incertidumbre; uno al 80 % que cubre el 88 % la sobreestima poco. El
cumplimiento se compara con el nivel nominal para que se vea la dirección del
error, no solo el número.

In [ ]:
iv = res['intervalos']
print('alpha estimado (NegBin):', iv['alpha'])

cob = pd.DataFrame(iv['cobertura']).T
print(cob.to_string())

print('Un cumplimiento < 1 significa que el intervalo es mas estrecho de lo')
print('que la cobertura real necesita: el error esta subestimado.')

## 6.4 · La tabla de afirmaciones

Igual que en el notebook 05: veredicto, evidencia y límite por afirmación, sin
combinar en un índice. El límite importa tanto como el veredicto: es lo que
impide que alguien use la afirmación fuera de su alcance.

In [ ]:
for nombre, a in res['veredicto']['afirmaciones'].items():
    print('=' * 78)
    print(nombre)
    print('  veredicto:', a['veredicto'])
    print('  evidencia:', a['evidencia'])
    print('  limite   :', a['limite'])
    print()
print(res['veredicto']['nota'])

## 6.5 · Cómo decirlo en una sustentación

> No: «el modelo predice los siniestros de la red vial». El holdout de 2025 no
> lo sostiene.

> Sí: «el modelo **ordena** los tramos por riesgo de forma útil y reproducible
> frente a la media de su región, tanto en validación espacial agrupada como en
> un holdout temporal que no vio; **no** estima bien el número de siniestros, y
> por eso el producto lo usa para priorizar, no para prognosticar cantidades».

## 6.6 · Reproducibilidad

```bash
python scripts/validez_predictiva.py        # regenera el JSON
python -m pytest tests/test_validez_predictiva.py -q
```

El informe completo está en `docs/validez_predictiva.md` y los datos crudos en
`data/processed/dashboard/validez_predictiva.json`.